In [35]:
import argparse
from datetime import datetime, timezone
import pandas as pd

from aind_data_schema.components.identifiers import Code, DataAsset
from aind_data_schema.core.processing import (
    DataProcess,
    Processing,
    ProcessName,
    ProcessStage,
    ResourceTimestamped,
    ResourceUsage,
)
from aind_data_schema_models.units import MemoryUnit
from aind_data_schema_models.system_architecture import OperatingSystem, CPUArchitecture
from codeocean import CodeOcean
import os
os.sys.path.append('/root/capsule/code/beh_ephys_analysis')
from utils.beh_functions import parseSessionID
import json

In [36]:
meta_data_file = '/root/capsule/code/data_management/meta_data_processing.xlsx'
session_meta = pd.read_excel(meta_data_file, sheet_name='sessions')
stan_meta = pd.read_excel(meta_data_file, sheet_name='animal_stan')
glm_meta = pd.read_excel(meta_data_file, sheet_name='animal_glm')
ccf_meta = pd.read_excel(meta_data_file, sheet_name='all_ccf')
dorsal_edge_meta = pd.read_excel(meta_data_file, sheet_name='all_dorsal_edge')
probe_location_meta = pd.read_excel(meta_data_file, sheet_name='all_probe_location')

computation_params_file = '/root/capsule/code/data_management/processing_params.json'
with open(computation_params_file, 'r') as f:
    computation_params = json.load(f)

In [37]:
session_data_files = ['/root/capsule/code/data_management/hopkins_session_assets.csv',
                        '/root/capsule/code/data_management/session_assets.csv',
                        '/root/capsule/code/data_management/hopkins_FP_session_assets.csv']
session_assets = pd.concat([pd.read_csv(file) for file in session_data_files], ignore_index=True)
attached_assets = pd.read_csv('/root/capsule/code/data_management/session_assets_sources.csv')

In [38]:
client = CodeOcean(domain="https://codeocean.allenneuraldynamics.org", token=os.getenv("API_SECRET"))
# %%
computation_id = os.getenv("CO_COMPUTATION_ID")
computation = client.computations.get_computation(computation_id=computation_id)

In [43]:
def create_session_meta(session_id):
    # prepare data asset names
    aniID, dateObj, raw_id = parseSessionID(session_id)
    session_dict = session_assets[session_assets['session_id'] == session_id].to_dict(orient='records')[0]
    session_data_assets_names = ['raw_data', 'sorted_curated']
    session_data_assets_ids = [session_dict[name] for name in session_data_assets_names if session_dict[name] is not None]
    session_data_assets_names_full = [client.data_assets.get_data_asset(data_asset_id=id).name for id in session_data_assets_ids]
    data_assets = [DataAsset(url=name) for name in session_data_assets_names_full]

    dp_list =[]
    session_output_folder = os.path.join('/root/capsule/scratch', aniID, session_id)
    if session_dict['sorted_curated'] is not None:
        curated_asset_name = client.data_assets.get_data_asset(data_asset_id=session_dict['sorted_curated']).name
        data_time = curated_asset_name.split('curated_')[-1]
        t = datetime.strptime(data_time, "%Y-%m-%d_%H-%M-%S").replace(tzinfo=timezone.utc)
    else:
        t = datetime(2026, 3, 27, 00, 00, 00, tzinfo=timezone.utc)
        
    for row_ind, row in session_meta.iterrows():
        output_folder = row['output'].format(aniID=aniID, session_id=session_id)
        # skip if not processed
        test_output = os.path.join(session_output_folder, output_folder)
        # print(f"Checking output: {test_output}")
        if os.path.isdir(test_output):
            if len(os.listdir(test_output))==0:
                # print(f"Output directory {test_output} is empty, skipping...")
                continue
        elif os.path.isfile(test_output):
            if not os.path.exists(test_output):
                # print(f"Output file {test_output} does not exist, skipping...")
                continue
        # check if there's parameter for this process
        if row['name'] in computation_params:
            params = computation_params[row['name']]
            # print(f"Using parameters for {row['name']}: {params}")
        else:
            params = {}
        curr_code = Code(
            url=row['url'],
            run_script=row['run_script'],
            version="0.1",
            parameters=params,
            input_data=data_assets
        )

        curr_dp = DataProcess(
                    process_type=ProcessName.OTHER,
                    name=row['name'],
                    experimenters=["Sue Su"],
                    stage=ProcessStage.ANALYSIS,
                    start_date_time=t,
                    end_date_time=t,
                    output_path=row['output'].format(aniID=aniID, session_id=session_id),
                    code=curr_code,
                    notes=row['name'] + '. ' + row['additional_note'] if pd.notna(row['additional_note']) else row['name'],
                    )
        dp_list.append(curr_dp)

    p = Processing.create_with_sequential_process_graph(
        data_processes=dp_list)
    return p

In [44]:
session_id = 'behavior_785956_2025-05-21_13-42-02'
p = create_session_meta(session_id)
    